In [1]:
%%capture
!pip install facenet-pytorch

## Libraries

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

In [3]:
import numpy as np
import cv2
import os
import time
import torch
import torchvision
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
from torch.utils.data import Subset
from image_iter import FaceDataset, customSubset


from utils import extract_face, take_picture, detect_crop_image, next_folder_name

%matplotlib inline
import matplotlib.pyplot as plt

/home/pj00/anaconda3/envs/CVenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

## Take image using camera OR Upload image

In [4]:
data_path = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/finetuning_data'

In [5]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [6]:
mtcnn = MTCNN(
    image_size=112,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device
)


In [7]:
transform = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
        ])

In [8]:
aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=.5, hue=.3),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)), 
    transforms.RandomAutocontrast(),
    transforms.ToPILImage()])

In [9]:
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


In [10]:
ss = customSubset(train_root)
idx_dixt = ss.generate_idx_dic()

In [11]:
def aug_img(save_path, transform, num_images):
    img = Image.open(save_path)
    partial_path = '/'.join(save_path.split('/')[:-1])
    for i in range(num_images-1):
        new_img = transform(img)
        new_img.save(os.path.join(partial_path, '{}.jpg'.format(i+1)))

In [12]:
def create_neg_class(path, num_images, transforms, dataset, idx_dict):
    keys = list(idx_dict.keys())
    
    for i in range(num_images):
        key = keys[i]
                
        dataset_sub = Subset(dataset, [idx_dict[key][0]])
        torch_img = dataset_sub.__getitem__(0)[0]
        pil_img = to_pil_image(torch_img)
        pil_img = transforms(pil_img)
#         print(os.path.join(path, '{}.jpg'.format(i)))
        pil_img.save(os.path.join(path, '{}.jpg'.format(i)))

In [14]:
num_classes = 1
num_images = 50
if num_classes == 1:
    path = os.path.join(data_path, '{}'.format(next_folder_name(data_path)))
    try:
        os.mkdir(path)
    except:
        print('Path exists: Issue!')
    create_neg_class(path, num_images, aug_transform, dataset, idx_dixt) # add 50 random images
    
    frame, save_path = extract_face(save_path=data_path)
    save_path = os.path.join(save_path, '0.jpg')
    images = detect_crop_image(frame=frame, model=mtcnn, transform=transform, device=device)
    aug_img(save_path=save_path, transform=aug_transform, num_images=50)
    
else:
    pass

[ WARN:0@88.110] global /croot/opencv-suite_1691620365762/work/modules/videoio/src/cap_gstreamer.cpp (862) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


Photo taken!
